In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 차원 축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
from sklearn.cluster import estimate_bandwidth

# 학습 모델 저장을 위한 라이브러리
import pickle

In [9]:
df11=pd.read_parquet(r'train/1.회원정보/201807_train_.parquet')
df12=pd.read_parquet(r'train/1.회원정보/201808_train_.parquet')
df13=pd.read_parquet(r'train/1.회원정보/201809_train_.parquet')
df14=pd.read_parquet(r'train/1.회원정보/201810_train_.parquet')
df15=pd.read_parquet(r'train/1.회원정보/201811_train_.parquet')
df16=pd.read_parquet(r'train/1.회원정보/201812_train_.parquet')

In [14]:
df11.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)

_2순위신용체크구분        157866
최종유효년월_신용_이용       79318
가입통신회사코드           67364
직장시도명              40807
최종유효년월_신용_이용가능     32828
최종카드발급일자            3007
_1순위신용체크구분          2147
dtype: int64

In [5]:
df_merged = pd.merge(df11, df12, on='ID', suffixes=('_7월', '_8월'))

In [6]:
import numpy as np

change_rate_df = pd.DataFrame()
change_rate_df['ID'] = df_merged['ID']

# 수치형 변수들에 대해서만
numerical_cols = [col.replace('_7월', '') for col in df_merged.columns if '_7월' in col and pd.api.types.is_numeric_dtype(df_merged[col])]

for col in numerical_cols:
    col_7 = f"{col}_7월"
    col_8 = f"{col}_8월"

    # 변화율 = (8월 - 7월) / abs(7월)
    change_rate_df[f"{col}_변화율"] = (df_merged[col_8] - df_merged[col_7]) / (np.abs(df_merged[col_7]) + 1e-6)

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import f_oneway

def compute_change_anova(df1, df2, key='ID', target_col='Segment', alpha=0.05):
    # ID 기준으로 병합
    merged = pd.merge(df1, df2, on=[key, target_col], suffixes=('_7', '_8'))
    
    # 수치형 컬럼만 선택
    num_cols = df1.select_dtypes(include=[np.number]).columns.difference([key])
    
    results = []
    for col in num_cols:
        col_7 = f"{col}_7"
        col_8 = f"{col}_8"
        if col_7 in merged.columns and col_8 in merged.columns:
            # 변화율 계산 (분모가 0인 경우 대비)
            with np.errstate(divide='ignore', invalid='ignore'):
                change = (merged[col_8] - merged[col_7]) / merged[col_7].replace(0, np.nan)
            
            merged['change_rate'] = change
            
            # Segment 그룹별로 ANOVA
            group_data = [
                merged.loc[merged[target_col] == seg, 'change_rate'].dropna()
                for seg in merged[target_col].unique()
            ]
            
            # 그룹 수 2 이상, 데이터가 모두 있는 경우만
            if len(group_data) >= 2 and all(len(g) > 1 for g in group_data):
                stat, p = f_oneway(*group_data)
                results.append({
                    '변수명': col,
                    'p-value': p,
                    '유의함': p < alpha
                })

    result_df = pd.DataFrame(results).sort_values(by='p-value')
    return result_df


In [6]:
# 모든 행/열 출력 설정
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [7]:
result_df1 = compute_change_anova(df11, df12)
display(result_df1)

,변수명,p-value,유의함
37,입회경과개월수_신용,0.000000e+00,True
33,이용카드수_신용,1.869196e-301,True
35,이용카드수_신용체크,3.616248e-283,True
31,이용여부_3M_해외겸용_본인,4.397582e-185,True
32,이용여부_3M_해외겸용_신용_본인,2.328154e-170,True
23,이용가능카드수_신용,4.740533e-136,True
17,유효카드수_신용,6.865518e-117,True
19,유효카드수_신용체크,3.226447e-103,True
0,_1순위카드이용건수,6.867703e-102,True
25,이용가능카드수_신용체크,1.594293e-79,True


In [8]:
result_df1.to_excel("01.회원정보(변화율).xlsx", index=False)
print('저장 완료')

저장 완료


In [2]:
import pandas as pd

def categorical_change_rate_per_id(df1, df2, id_col='ID'):
    # 범주형 변수만 추출 (숫자형 제외)
    cat_cols = df1.select_dtypes(exclude=['number']).columns.tolist()
    
    # ID 컬럼은 제외
    cat_cols = [col for col in cat_cols if col != id_col]
    
    # 두 데이터프레임에서 필요한 컬럼만 추출
    df1_sub = df1[[id_col] + cat_cols]
    df2_sub = df2[[id_col] + cat_cols]
    
    # 병합
    merged = pd.merge(df1_sub, df2_sub, on=id_col, suffixes=('_prev', '_curr'))
    
    # 변화율 계산
    change_rates = {}
    for col in cat_cols:
        changed = (merged[f"{col}_prev"] != merged[f"{col}_curr"]).mean()
        change_rates[col] = round(changed, 4)
    
    # 결과를 데이터프레임으로 정리
    result_df = pd.DataFrame({
        '컬럼명': list(change_rates.keys()),
        'ID 기준 변화율': list(change_rates.values())
    }).sort_values(by='ID 기준 변화율', ascending=False).reset_index(drop=True)
    
    return result_df


In [19]:
result_df = categorical_change_rate_per_id(df11, df12)
display(result_df)

,컬럼명,ID 기준 변화율
0,_2순위신용체크구분,0.3985
1,직장시도명,0.1858
2,가입통신회사코드,0.1709
3,거주시도명,0.0552
4,_1순위신용체크구분,0.0120
5,연회비발생카드수_B0M,0.0116
6,Life_Stage,0.0031
7,Segment,0.0000
8,연령,0.0000
9,상품관련면제카드수_B0M,0.0000


In [21]:
result_df = categorical_change_rate_per_id(df21, df22)
display(result_df)

,컬럼명,ID 기준 변화율
0,RV전환가능여부,0.0202
1,자발한도감액횟수_R12M,0.0018
2,한도증액횟수_R12M,0.0011
3,카드론동의여부,0.0011
4,한도심사요청건수,0.0001


---

In [15]:
df21=pd.read_parquet(r'train/2.신용정보/201807_train_.parquet')
df22=pd.read_parquet(r'train/2.신용정보/201808_train_.parquet')
df23=pd.read_parquet(r'train/2.신용정보/201809_train_.parquet')
df24=pd.read_parquet(r'train/2.신용정보/201810_train_.parquet')
df25=pd.read_parquet(r'train/2.신용정보/201811_train_.parquet')
df26=pd.read_parquet(r'train/2.신용정보/201812_train_.parquet')

In [16]:
df21.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)

RV신청일자    325079
dtype: int64

In [10]:
tg1=pd.read_parquet(r'train/1.회원정보/201807_train_.parquet')
tg2=pd.read_parquet(r'train/1.회원정보/201808_train_.parquet')
tg3=pd.read_parquet(r'train/1.회원정보/201809_train_.parquet')
tg4=pd.read_parquet(r'train/1.회원정보/201810_train_.parquet')
tg5=pd.read_parquet(r'train/1.회원정보/201811_train_.parquet')
tg6=pd.read_parquet(r'train/1.회원정보/201812_train_.parquet')

In [11]:
df_merged2 = pd.merge(df21, df22, on='ID', suffixes=('_7월', '_8월'))

In [13]:
tg1 = pd.read_parquet(r'train/1.회원정보/201807_train_.parquet')
tg2 = pd.read_parquet(r'train/1.회원정보/201808_train_.parquet')

tg_df = pd.concat([tg1, tg2], ignore_index=True)

In [17]:
df21['Segment'] = tg1['Segment'].values
df22['Segment'] = tg2['Segment'].values

In [15]:
df_merged2['Segment'] = tg1['Segment'].values

In [18]:
result_df2 = compute_change_anova(df21, df22)
display(result_df2)

,변수명,p-value,유의함
26,한도증액후경과월,6.358981e-182,True
6,RV최소결제비율,5.382711e-121,True
11,강제한도감액후경과월,2.970899e-119,True
13,상향가능CA한도금액,1.030350e-69,True
21,카드이용한도금액_B1M,2.453686e-57,True
16,일시불ONLY전환가능여부,5.019215e-52,True
25,한도증액금액_R12M,3.721745e-49,True
0,CA이자율_할인전,7.007814e-24,True
14,상향가능한도금액,2.392432e-23,True
9,강제한도감액금액_R12M,7.065061e-22,True


In [19]:
result_df2.to_excel("02.신용정보(변화율).xlsx", index=False)
print('저장 완료')

저장 완료


In [ ]:
result_df = categorical_change_rate_per_id(df21, df22)
display(result_df)

---

### 3번 데이터 (70까지)

In [2]:
df311=pd.read_parquet(r'train/3.승인매출정보/201807_train_.parquet').iloc[:, :70]
df312=pd.read_parquet(r'train/3.승인매출정보/201808_train_.parquet').iloc[:, :70]
df313=pd.read_parquet(r'train/3.승인매출정보/201809_train_.parquet').iloc[:, :70]
df314=pd.read_parquet(r'train/3.승인매출정보/201810_train_.parquet').iloc[:, :70]
df315=pd.read_parquet(r'train/3.승인매출정보/201811_train_.parquet').iloc[:, :70]
df316=pd.read_parquet(r'train/3.승인매출정보/201812_train_.parquet').iloc[:, :70]

In [3]:
df311.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [46]:
df_merged31 = pd.merge(df311, df312, on='ID', suffixes=('_7월', '_8월'))

In [47]:
df311['Segment'] = tg1['Segment'].values
df312['Segment'] = tg2['Segment'].values

In [48]:
result_df31 = compute_change_anova(df311, df312)
display(result_df31)

,변수명,p-value,유의함
11,이용건수_신판_B0M,0.000000e+00,True
13,이용건수_일시불_B0M,0.000000e+00,True
9,이용건수_신용_B0M,0.000000e+00,True
35,이용후경과월_CA,0.000000e+00,True
3,이용개월수_일시불_R12M,8.478470e-289,True
32,이용금액_할부_무이자_B0M,7.144614e-287,True
30,이용금액_할부_B0M,1.005287e-268,True
14,이용건수_일시불_R12M,3.709533e-259,True
20,이용건수_할부_무이자_B0M,3.314648e-249,True
1,이용개월수_신용_R12M,3.239766e-246,True


In [49]:
result_df31.to_excel("03.승인매출정보(변화율)(1).xlsx", index=False)
print('저장 완료')

저장 완료


In [23]:
result_df = categorical_change_rate_per_id(df311, df312)
display(result_df)

,컬럼명,ID 기준 변화율


### 3번 데이터 (71 ~ 140까지)

In [4]:
df321=pd.read_parquet(r'train/3.승인매출정보/201807_train_.parquet').iloc[:, 71:140]
df322=pd.read_parquet(r'train/3.승인매출정보/201808_train_.parquet').iloc[:, 71:140]
df323=pd.read_parquet(r'train/3.승인매출정보/201809_train_.parquet').iloc[:, 71:140]
df324=pd.read_parquet(r'train/3.승인매출정보/201810_train_.parquet').iloc[:, 71:140]
df325=pd.read_parquet(r'train/3.승인매출정보/201811_train_.parquet').iloc[:, 71:140]
df326=pd.read_parquet(r'train/3.승인매출정보/201812_train_.parquet').iloc[:, 71:140]

In [5]:
df321.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [13]:
df321['ID']=df311['ID'].values
df322['ID']=df312['ID'].values

In [52]:
df_merged32 = pd.merge(df321, df322, on='ID', suffixes=('_7월', '_8월'))

In [53]:
df321['Segment'] = tg1['Segment'].values
df322['Segment'] = tg2['Segment'].values

In [54]:
result_df32 = compute_change_anova(df321, df322)
display(result_df32)

,변수명,p-value,유의함
22,이용개월수_할부_R3M,4.821067e-202,True
24,이용개월수_할부_무이자_R3M,2.582792e-187,True
33,이용건수_일시불_R6M,9.115426e-183,True
17,이용개월수_일시불_R6M,1.245099e-163,True
15,이용개월수_신판_R6M,3.298726e-157,True
29,이용건수_신용_R6M,9.091406e-152,True
13,이용개월수_신용_R6M,1.319366e-135,True
31,이용건수_신판_R6M,7.023265e-131,True
8,쇼핑_편의점_이용금액,4.642911e-100,True
32,이용건수_일시불_R3M,7.214742e-90,True


In [55]:
result_df32.to_excel("03.승인매출정보(변화율)(2).xlsx", index=False)
print('저장 완료')

저장 완료


In [14]:
result_df = categorical_change_rate_per_id(df321, df322)
display(result_df)

,컬럼명,ID 기준 변화율


### 3번 데이터 (141 ~ 210번까지)

In [6]:
df331=pd.read_parquet(r'train/3.승인매출정보/201807_train_.parquet').iloc[:, 141:210]
df332=pd.read_parquet(r'train/3.승인매출정보/201808_train_.parquet').iloc[:, 141:210]
df333=pd.read_parquet(r'train/3.승인매출정보/201809_train_.parquet').iloc[:, 141:210]
df334=pd.read_parquet(r'train/3.승인매출정보/201810_train_.parquet').iloc[:, 141:210]
df335=pd.read_parquet(r'train/3.승인매출정보/201811_train_.parquet').iloc[:, 141:210]
df336=pd.read_parquet(r'train/3.승인매출정보/201812_train_.parquet').iloc[:, 141:210]

In [7]:
df331.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)

_3순위여유업종    390745
_3순위납부업종    384482
_2순위여유업종    366992
_2순위납부업종    336229
_3순위교통업종    335842
_1순위여유업종    297862
_2순위교통업종    269582
_3순위쇼핑업종    215290
_1순위납부업종    196673
_1순위교통업종    186099
_2순위쇼핑업종    184045
_3순위업종      175156
_1순위쇼핑업종    143665
_2순위업종      141819
_1순위업종       79580
dtype: int64

In [15]:
df331['ID']=df311['ID'].values
df332['ID']=df312['ID'].values

In [58]:
df_merged33 = pd.merge(df331, df332, on='ID', suffixes=('_7월', '_8월'))

In [59]:
df331['Segment'] = tg1['Segment'].values
df332['Segment'] = tg2['Segment'].values

In [60]:
result_df33 = compute_change_anova(df331, df332)
display(result_df33)

,변수명,p-value,유의함
4,_1순위여유업종_이용금액,7.493249e-66,True
2,_1순위쇼핑업종_이용금액,2.639937e-63,True
17,교통_택시이용금액,2.795764e-56,True
25,여유_숙박이용금액,2.916640e-42,True
12,_3순위쇼핑업종_이용금액,1.409398e-37,True
7,_2순위쇼핑업종_이용금액,1.680973e-35,True
16,교통_철도버스이용금액,2.619285e-33,True
26,여유_운동이용금액,3.691897e-30,True
38,할부금액_유이자_14M_R12M,1.050789e-28,True
24,여유_기타이용금액,2.624450e-19,True


In [61]:
result_df33.to_excel("03.승인매출정보(변화율)(3).xlsx", index=False)
print('저장 완료')

저장 완료


In [16]:
result_df = categorical_change_rate_per_id(df331, df332)
display(result_df)

,컬럼명,ID 기준 변화율
0,_3순위여유업종,0.9907
1,_3순위납부업종,0.9643
2,_2순위여유업종,0.9539
3,_3순위교통업종,0.8819
4,_2순위납부업종,0.8548
5,_1순위여유업종,0.7991
6,_2순위교통업종,0.7250
7,_3순위쇼핑업종,0.6834
8,_3순위업종,0.6408
9,_2순위쇼핑업종,0.6045


### 3번 데이터 (211 ~ 280번까지)

In [17]:
df341=pd.read_parquet(r'train/3.승인매출정보/201807_train_.parquet').iloc[:, 211:280]
df342=pd.read_parquet(r'train/3.승인매출정보/201808_train_.parquet').iloc[:, 211:280]
df343=pd.read_parquet(r'train/3.승인매출정보/201809_train_.parquet').iloc[:, 211:280]
df344=pd.read_parquet(r'train/3.승인매출정보/201810_train_.parquet').iloc[:, 211:280]
df345=pd.read_parquet(r'train/3.승인매출정보/201811_train_.parquet').iloc[:, 211:280]
df346=pd.read_parquet(r'train/3.승인매출정보/201812_train_.parquet').iloc[:, 211:280]

In [18]:
df341['ID']=df311['ID'].values
df342['ID']=df312['ID'].values

In [64]:
df_merged34 = pd.merge(df341, df342, on='ID', suffixes=('_7월', '_8월'))

In [65]:
df341['Segment'] = tg1['Segment'].values
df342['Segment'] = tg2['Segment'].values

In [66]:
result_df34 = compute_change_anova(df341, df342)
display(result_df34)

,변수명,p-value,유의함
33,최종카드론이용경과월,1.552719e-246,True
9,RP후경과월_교통,6.705255e-221,True
18,이용개월수_오프라인_R6M,1.936811e-134,True
22,이용금액_오프라인_R6M,7.270554e-70,True
21,이용건수_온라인_R6M,3.116811e-59,True
6,RP후경과월,1.907899e-54,True
1,RP건수_교통_B0M,3.166119e-22,True
0,RP건수_B0M,2.000225e-12,True
26,증감_RP유형건수_전월,1.255431e-11,True
4,RP금액_B0M,2.639676e-11,True


In [67]:
result_df34.to_excel("03.승인매출정보(변화율)(4).xlsx", index=False)
print('저장 완료')

저장 완료


In [19]:
result_df = categorical_change_rate_per_id(df341, df342)
display(result_df)

,컬럼명,ID 기준 변화율
0,최종카드론_신청경로코드,0.8281


### 3번 데이터 (281~350번까지)

In [5]:
df351=pd.read_parquet(r'train/3.승인매출정보/201807_train_.parquet').iloc[:, 281:350]
df352=pd.read_parquet(r'train/3.승인매출정보/201808_train_.parquet').iloc[:, 281:350]
df353=pd.read_parquet(r'train/3.승인매출정보/201809_train_.parquet').iloc[:, 281:350]
df354=pd.read_parquet(r'train/3.승인매출정보/201810_train_.parquet').iloc[:, 281:350]
df355=pd.read_parquet(r'train/3.승인매출정보/201811_train_.parquet').iloc[:, 281:350]
df356=pd.read_parquet(r'train/3.승인매출정보/201812_train_.parquet').iloc[:, 281:350]

In [6]:
df351['ID']=df311['ID'].values
df352['ID']=df312['ID'].values

In [70]:
df_merged35 = pd.merge(df351, df352, on='ID', suffixes=('_7월', '_8월'))

In [71]:
df351['Segment'] = tg1['Segment'].values
df352['Segment'] = tg2['Segment'].values

In [72]:
result_df35 = compute_change_anova(df351, df352)
display(result_df35)

,변수명,p-value,유의함
25,이용건수_페이_온라인_B0M,2.151957e-189,True
20,이용건수_온라인_B0M,3.352723e-189,True
36,이용금액_간편결제_B0M,1.528315e-188,True
46,이용금액_페이_온라인_R3M,4.639596e-180,True
45,이용금액_페이_온라인_B0M,4.233471e-171,True
21,이용건수_온라인_R3M,4.343630e-123,True
41,이용금액_온라인_B0M,2.448343e-121,True
37,이용금액_간편결제_R3M,9.536722e-115,True
17,이용건수_간편결제_R6M,5.240548e-66,True
3,이용개월수_간편결제_R6M,5.243673e-45,True


In [73]:
result_df35.to_excel("03.승인매출정보(변화율)(5).xlsx", index=False)
print('저장 완료')

저장 완료


In [7]:
result_df = categorical_change_rate_per_id(df351, df352)
display(result_df)

,컬럼명,ID 기준 변화율


### 3번 데이터 (351~)

In [8]:
df361=pd.read_parquet(r'train/3.승인매출정보/201807_train_.parquet').iloc[:, 351:]
df362=pd.read_parquet(r'train/3.승인매출정보/201808_train_.parquet').iloc[:, 351:]
df363=pd.read_parquet(r'train/3.승인매출정보/201809_train_.parquet').iloc[:, 351:]
df364=pd.read_parquet(r'train/3.승인매출정보/201810_train_.parquet').iloc[:, 351:]
df365=pd.read_parquet(r'train/3.승인매출정보/201811_train_.parquet').iloc[:, 351:]
df366=pd.read_parquet(r'train/3.승인매출정보/201812_train_.parquet').iloc[:, 351:]

In [9]:
df361['ID']=df311['ID'].values
df362['ID']=df312['ID'].values

In [90]:
df_merged36 = pd.merge(df361, df362, on='ID', suffixes=('_7월', '_8월'))

In [91]:
df361['Segment'] = tg1['Segment'].values
df362['Segment'] = tg2['Segment'].values

In [92]:
result_df36 = compute_change_anova(df361, df362)
display(result_df36)

,변수명,p-value,유의함
21,이용개월수_전체_R6M,8.801242e-175,True
13,연속유실적개월수_기본_24M_카드,4.974024e-154,True
34,정상입금원금_B0M,3.619375e-134,True
18,이용개월수_결제일_R6M,2.335739e-126,True
11,신청건수_ATM_CA_B0,5.992639e-77,True
9,승인거절건수_기타_R3M,1.073505e-73,True
35,정상입금원금_B2M,5.983262e-65,True
36,정상입금원금_B5M,4.723985e-60,True
4,선입금원금_B0M,1.670703e-30,True
17,이용개월수_결제일_R3M,5.918141e-21,True


In [93]:
result_df36.to_excel("03.승인매출정보(변화율)(6).xlsx", index=False)
print('저장 완료')

저장 완료


In [10]:
result_df = categorical_change_rate_per_id(df361, df362)
display(result_df)

,컬럼명,ID 기준 변화율
0,이용금액대,0.1029


---

In [8]:
df41=pd.read_parquet(r'train/4.청구입금정보/201807_train_.parquet')
df42=pd.read_parquet(r'train/4.청구입금정보/201808_train_.parquet')
df43=pd.read_parquet(r'train/4.청구입금정보/201809_train_.parquet')
df44=pd.read_parquet(r'train/4.청구입금정보/201810_train_.parquet')
df45=pd.read_parquet(r'train/4.청구입금정보/201811_train_.parquet')
df46=pd.read_parquet(r'train/4.청구입금정보/201812_train_.parquet')

In [9]:
df41.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [95]:
df_merged4 = pd.merge(df41, df42, on='ID', suffixes=('_7월', '_8월'))

In [96]:
df41['Segment'] = tg1['Segment'].values
df42['Segment'] = tg2['Segment'].values

In [97]:
result_df4 = compute_change_anova(df41, df42)
display(result_df4)

,변수명,p-value,유의함
16,청구서발송여부_B0,0.000000e+00,True
17,청구서발송여부_R3M,4.997803e-199,True
18,청구서발송여부_R6M,1.624499e-132,True
23,포인트_마일리지_환산_B0M,4.470884e-28,True
14,청구금액_R3M,1.733910e-26,True
30,포인트_포인트_월적립_R3M,1.969592e-25,True
8,상환개월수_결제일_R6M,1.751499e-24,True
5,마일_적립포인트_R12M,9.500996e-17,True
31,할인금액_B0M,1.270836e-16,True
25,포인트_이용포인트_R3M,6.250139e-15,True


In [104]:
result_df4.to_excel("04.청구입금정보.xlsx", index=False)
print('저장 완료')

저장 완료


In [12]:
result_df = categorical_change_rate_per_id(df41, df42)
display(result_df)

,컬럼명,ID 기준 변화율
0,할인건수_R3M,0.0300
1,대표청구서수령지구분코드,0.0157
2,청구서수령방법,0.0157
3,대표청구지고객주소구분코드,0.0054
4,할인건수_B0M,0.0025
5,대표결제방법코드,0.0000


---

In [10]:
df51=pd.read_parquet(r'train/5.잔액정보/201807_train_.parquet')
df52=pd.read_parquet(r'train/5.잔액정보/201808_train_.parquet')
df53=pd.read_parquet(r'train/5.잔액정보/201809_train_.parquet')
df54=pd.read_parquet(r'train/5.잔액정보/201810_train_.parquet')
df55=pd.read_parquet(r'train/5.잔액정보/201811_train_.parquet')
df56=pd.read_parquet(r'train/5.잔액정보/201812_train_.parquet')

In [11]:
df51.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)

연체일자_B0M    398300
dtype: int64

In [100]:
df_merged5 = pd.merge(df51, df52, on='ID', suffixes=('_7월', '_8월'))

In [101]:
df51['Segment'] = tg1['Segment'].values
df52['Segment'] = tg2['Segment'].values

In [102]:
result_df5 = compute_change_anova(df51, df52)
display(result_df5)

,변수명,p-value,유의함
20,잔액_일시불_B0M,8.037874e-159,True
36,잔액_현금서비스_B2M,1.160159e-137,True
7,연체일수_B1M,5.990399e-114,True
32,잔액_할부_무이자_B0M,8.544107e-89,True
34,잔액_현금서비스_B0M,5.831520e-77,True
37,최종연체회차,1.175484e-65,True
31,잔액_할부_B2M,3.737798e-54,True
43,평잔_RV일시불_6M,9.661154e-42,True
30,잔액_할부_B1M,1.030501e-41,True
13,월중평잔_RV일시불,1.318603e-32,True


In [105]:
result_df5.to_excel("05.잔액정보.xlsx", index=False)
print('저장 완료')

저장 완료


In [15]:
result_df = categorical_change_rate_per_id(df51, df52)
display(result_df)

,컬럼명,ID 기준 변화율


---

In [12]:
df61=pd.read_parquet(r'train/6.채널정보/201807_train_.parquet')
df62=pd.read_parquet(r'train/6.채널정보/201808_train_.parquet')
df63=pd.read_parquet(r'train/6.채널정보/201809_train_.parquet')
df64=pd.read_parquet(r'train/6.채널정보/201810_train_.parquet')
df65=pd.read_parquet(r'train/6.채널정보/201811_train_.parquet')
df66=pd.read_parquet(r'train/6.채널정보/201812_train_.parquet')

In [13]:
df61.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)

OS구분코드    272261
dtype: int64

In [107]:
df_merged6 = pd.merge(df61, df62, on='ID', suffixes=('_7월', '_8월'))

In [108]:
df61['Segment'] = tg1['Segment'].values
df62['Segment'] = tg2['Segment'].values

In [109]:
result_df6 = compute_change_anova(df61, df62)
display(result_df6)

,변수명,p-value,유의함
15,방문후경과월_PC_R6M,0.000000e+00,True
35,인입후경과월_IB_R6M,0.000000e+00,True
21,이용메뉴건수_ARS_B0M,5.877203e-190,True
31,인입횟수_ARS_B0M,2.150864e-185,True
16,방문후경과월_모바일웹_R6M,3.954263e-165,True
27,인입일수_ARS_B0M,8.896239e-142,True
19,상담건수_B0M,1.378151e-45,True
32,인입횟수_IB_B0M,2.815416e-45,True
17,방문후경과월_앱_R6M,3.118893e-41,True
22,이용메뉴건수_IB_B0M,5.426736e-25,True


In [110]:
result_df6.to_excel("06.채널정보.xlsx", index=False)
print('저장 완료')

저장 완료


In [4]:
result_df = categorical_change_rate_per_id(df61, df62)
display(result_df)

,컬럼명,ID 기준 변화율
0,OS구분코드,0.6807
1,방문횟수_앱_R6M,0.0598
2,방문횟수_PC_R6M,0.0258
3,방문일수_PC_R6M,0.0186
4,이용메뉴건수_ARS_R6M,0.0059
5,인입횟수_ARS_R6M,0.0051


---

In [5]:
df71=pd.read_parquet(r'train/7.마케팅정보/201807_train_.parquet')
df72=pd.read_parquet(r'train/7.마케팅정보/201808_train_.parquet')
df73=pd.read_parquet(r'train/7.마케팅정보/201809_train_.parquet')
df74=pd.read_parquet(r'train/7.마케팅정보/201810_train_.parquet')
df75=pd.read_parquet(r'train/7.마케팅정보/201811_train_.parquet')
df76=pd.read_parquet(r'train/7.마케팅정보/201812_train_.parquet')

In [112]:
df_merged7 = pd.merge(df71, df72, on='ID', suffixes=('_7월', '_8월'))

In [113]:
df71['Segment'] = tg1['Segment'].values
df72['Segment'] = tg2['Segment'].values

In [114]:
result_df7 = compute_change_anova(df71, df72)
display(result_df7)

,변수명,p-value,유의함
11,컨택건수_이용유도_인터넷_B0M,0.000000e+00,True
8,컨택건수_이용유도_LMS_B0M,3.213133e-183,True
14,컨택건수_이용유도_청구서_R6M,5.467036e-137,True
5,컨택건수_부대서비스_TM_R6M,1.776933e-45,True
13,컨택건수_이용유도_청구서_B0M,1.452570e-26,True
9,컨택건수_이용유도_LMS_R6M,6.073298e-25,True
18,컨택건수_카드론_TM_R6M,1.327176e-13,True
10,컨택건수_이용유도_TM_R6M,9.619041e-12,True
12,컨택건수_이용유도_인터넷_R6M,1.021604e-11,True
7,컨택건수_이용유도_EM_R6M,2.188299e-10,True


In [115]:
result_df7.to_excel("07.마케팅정보.xlsx", index=False)
print('저장 완료')

저장 완료


In [6]:
result_df = categorical_change_rate_per_id(df71, df72)
display(result_df)

,컬럼명,ID 기준 변화율
0,캠페인접촉건수_R12M,0.0983
1,캠페인접촉일수_R12M,0.0909


---

In [7]:
df81=pd.read_parquet(r'train/8.성과정보/201807_train_.parquet')
df82=pd.read_parquet(r'train/8.성과정보/201808_train_.parquet')
df83=pd.read_parquet(r'train/8.성과정보/201809_train_.parquet')
df84=pd.read_parquet(r'train/8.성과정보/201810_train_.parquet')
df85=pd.read_parquet(r'train/8.성과정보/201811_train_.parquet')
df86=pd.read_parquet(r'train/8.성과정보/201812_train_.parquet')

In [118]:
df_merged8 = pd.merge(df81, df82, on='ID', suffixes=('_7월', '_8월'))

In [117]:
df81['Segment'] = tg1['Segment'].values
df82['Segment'] = tg2['Segment'].values

In [119]:
result_df8 = compute_change_anova(df81, df82)
display(result_df8)

,변수명,p-value,유의함
28,증감율_이용건수_할부_전월,4.446591e-323,True
1,변동률_CA평잔,1.275082e-91,True
8,변동률_카드론평잔,3.127545e-40,True
18,증감율_이용건수_CA_분기,1.528786e-25,True
9,변동률_할부평잔,8.187084e-16,True
3,변동률_RV일시불평잔,9.347657e-11,True
4,변동률_일시불평잔,1.114251e-08,True
2,변동률_RVCA평잔,6.490196e-06,True
27,증감율_이용건수_할부_분기,6.514682e-06,True
12,잔액_신판ca평균한도소진율_r3m,5.152581e-03,True


In [120]:
result_df8.to_excel("08.마케팅정보.xlsx", index=False)
print('저장 완료')

저장 완료


In [8]:
result_df = categorical_change_rate_per_id(df81, df82)
display(result_df)

,컬럼명,ID 기준 변화율


---

In [122]:
# 결과 데이터프레임들을 리스트에 넣고 수직 방향으로 합치기
all_results = pd.concat(
    [result_df1, result_df2, result_df31, result_df32,
     result_df33, result_df34, result_df35, result_df36,
     result_df4, result_df5, result_df6, result_df7, result_df8],
    axis=0,
    ignore_index=True
)

# 결과 출력
display(all_results)


,변수명,p-value,유의함
0,입회경과개월수_신용,0.000000e+00,True
1,이용카드수_신용,1.869196e-301,True
2,이용카드수_신용체크,3.616248e-283,True
3,이용여부_3M_해외겸용_본인,4.397582e-185,True
4,이용여부_3M_해외겸용_신용_본인,2.328154e-170,True
5,이용가능카드수_신용,4.740533e-136,True
6,유효카드수_신용,6.865518e-117,True
7,유효카드수_신용체크,3.226447e-103,True
8,_1순위카드이용건수,6.867703e-102,True
9,이용가능카드수_신용체크,1.594293e-79,True


In [123]:
# 유의하지 않은 컬럼만 필터링
not_significant_cols = all_results[all_results['유의함'] == False]['변수명'].tolist()

# 결과 확인
print(not_significant_cols)

['최종유효년월_신용_이용가능', '최종유효년월_신용_이용', '카드신청건수', '최종카드발급일자', '_2순위카드이용건수', '이용금액_R3M_신용', '유효카드수_체크', '이용가능카드수_체크', '이용금액_R3M_신용체크', '_2순위카드이용금액', '_1순위카드이용금액', '기준년월', '남녀구분코드', '입회일자_신용', 'RV신청일자', '한도심사요청후경과월', 'RV일시불이자율_할인전', '자발한도감액후경과월', '기준년월', '최초한도금액', '최대이용금액_할부_R12M', '이용개월수_할부_R12M', '이용건수_할부_R12M', '이용건수_카드론_R12M', '이용건수_할부_유이자_R12M', '최대이용금액_카드론_R12M', '최대이용금액_할부_유이자_R12M', '이용개월수_할부_무이자_R12M', '이용금액_체크_B0M', '최종이용일자_체크', '이용금액_일시불_B0M', '이용개월수_할부_유이자_R12M', '최종이용일자_카드론', '최종이용일자_CA', '이용금액_체크_R12M', '이용금액_일시불_R12M', '기준년월', '이용개월수_CA_R6M', '이용금액_CA_R3M', '이용금액_체크_R3M', '이용개월수_체크_R6M', '이용금액_할부_무이자_R3M', '이용금액_할부_R3M', '이용금액_일시불_R3M', '이용금액_할부_무이자_R6M', '이용금액_할부_R6M', '이용금액_일시불_R6M', '이용금액_체크_R6M', '_1순위납부업종_이용금액', '납부_보험료이용금액', '교통_버스지하철이용금액', '_3순위교통업종_이용금액', '할부건수_3M_R12M', '할부건수_유이자_3M_R12M', '_3순위업종_이용금액', '납부_통신비이용금액', '_2순위납부업종_이용금액', '_1순위업종_이용금액', '_3순위납부업종_이용금액', '_2순위업종_이용금액', '할부금액_12M_R12M', '할부건수_유이자_6M_R12M', '증감_RP건수_전월', 'RP후경과월_학습비', 'RP후경과월_렌탈', 'RP건수_보험_B0

In [124]:
# 유의하지 않은 컬럼만 추출한 데이터프레임
not_significant_df = all_results[all_results['유의함'] == False].reset_index(drop=True)

# 결과 출력
display(not_significant_df)

,변수명,p-value,유의함
0,최종유효년월_신용_이용가능,0.077465,False
1,최종유효년월_신용_이용,0.128899,False
2,카드신청건수,0.349883,False
3,최종카드발급일자,0.369289,False
4,_2순위카드이용건수,0.479684,False
5,이용금액_R3M_신용,0.587969,False
6,유효카드수_체크,0.635999,False
7,이용가능카드수_체크,0.699449,False
8,이용금액_R3M_신용체크,0.715127,False
9,_2순위카드이용금액,0.899265,False


In [125]:
not_significant_df.to_excel("데이터 변화율 종합.xlsx", index=False)
print('저장 완료')

저장 완료
